# Translate the 5k sample into English

**What this notebook does:** it takes the 5,000 sampled paragraphs from `sample_for_llm.jsonl` (made by `sample5k.ipynb`) and translates each one into English with the uni AI, using the professor's translation prompt.

**Before you start**
1. Run **`sample5k.ipynb`** first (in the main folder). It creates `sample_for_llm.jsonl`. It needs the data file `all_multi_paragraphs_2022_2026.RDS` in the main folder.
2. Kernel in the top right: **`.venv`**.

**Order:** STEP 1 → STEP 6. Run the **test batch (STEP 4)** first, check it, and only then start the **full run (STEP 5)**.

**Safe to stop:** every finished paragraph is saved to `translation_checkpoint.jsonl` straight away. If VS Code crashes or you stop the run, just run STEP 1–3 again and then STEP 5. It **skips everything already translated** and continues.

---
## STEP 1: Settings (connect to the university AI)

A box appears at the **top centre of VS Code**: paste the API key from `Howto.rmd` (starts with `sk-`).

`MAX_WORKERS` = how many paragraphs are sent **at the same time**. 4 is a careful start. If there are no errors, you can try 8.

In [ ]:
import os, json, re, time, getpass, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
import pandas as pd

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "gemma4-26b"
TEMPERATURE = 0
SEED = 123
MAX_WORKERS = 8                     # parallel calls (like parallel_calls in bacchuss_lyaeus); STEP 4b finds the best value

SAMPLE_FILE     = "../sample_for_llm.jsonl"        # output of sample5k.ipynb
CHECKPOINT_FILE = "translation_checkpoint.jsonl"   # grows while translating
OUTPUT_XLSX     = "sample_5k_translated.xlsx"
OUTPUT_JSONL    = "sample_5k_translated.jsonl"

API_KEY = os.environ.get("UNI_API_KEY") or getpass.getpass("Paste the Uni API key: ")
print("Key loaded:", bool(API_KEY))

---
## STEP 2: The professor's translation prompt

Same prompt as in `translation_test.ipynb`. **Don't change it.**

In [ ]:
PROFESSOR_PROMPT = """You are a careful and attentive translator for news articles.

You will receive a multiparagraph in this form:

Text: <multi-paragraph>

You are given a multi-paragraph from a news article and translate it into clear English.
Preserve names and numbers exactly, translate quotes as directly as possible.
Return only the translation.
Do not comment on the translation.

If paragraphs contain sentence fragments, assume it is a headline or image description. Translate them as sentence fragments as well. Maintain the paragraph structure.

Some entity names will be masked as [Gruppe 1] and [andere Gruppe]. Maintain those names.

Output Format:

Return only a valid JSON object:

{
  "label": "<translated text>"
}

No additional explanations, no Markdown, and no text outside the JSON."""

instructions = PROFESSOR_PROMPT
reminder = None   # the professor gave no reminder for translation
JSON_MODE = True  # the prompt asks for JSON output
print(instructions)

---
## STEP 3: Load the 5k sample

**✅ You should see:** `5000 paragraphs loaded` and the first rows.

Each paragraph gets a `row_id` (0–4999) so we can match every translation to its paragraph, even after a restart.

In [ ]:
sample = pd.read_json(SAMPLE_FILE, lines=True)
sample.insert(0, "row_id", range(len(sample)))
print(f"{len(sample)} paragraphs loaded")
print("Columns:", list(sample.columns))
print("Paragraphs with [Gruppe 1]:", sample["text_block"].str.contains(r"\[Gruppe 1\]").sum())
sample.head(3)

---
## Helper functions (run, nothing to change)

- `call_llm`: sends one paragraph to the AI (same message as `bacchuss`)
- `translate_row`: translates one paragraph, reads the JSON and runs quick checks
- `run_batch`: translates many paragraphs in parallel and saves each result to the checkpoint file

In [ ]:
def call_llm(text, instructions, reminder=None, json_mode=False, max_retries=3):
    user_msg = f"Text: '{text}'"
    if reminder:
        user_msg += f"\nReminder: {reminder}"
    body = {
        "model": MODEL,
        "messages": [{"role": "system", "content": instructions},
                     {"role": "user",   "content": user_msg}],
        "temperature": TEMPERATURE,
        "seed": SEED,
        "reasoning_effort": "none",
    }
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(f"{HOST}/chat/completions",
                              headers={"Authorization": f"Bearer {API_KEY}"},
                              json=body, timeout=180)
            r.raise_for_status()
            j = r.json()
            return {"output": j["choices"][0]["message"]["content"].strip(),
                    "tokens_prompt": j["usage"]["prompt_tokens"],
                    "tokens_response": j["usage"]["completion_tokens"],
                    "error": None}
        except Exception as e:
            last_error = str(e)[:300]
            time.sleep(5 * attempt)          # wait longer after each failed try
    return {"output": None, "tokens_prompt": None, "tokens_response": None, "error": last_error}


PLACEHOLDER = re.compile(r"\[(Gruppe 1|Andere Gruppe|andere Gruppe)\]")

def translate_row(row_id, text):
    start = time.time()
    res = call_llm(text, instructions, reminder, json_mode=JSON_MODE)
    raw = res["output"]
    try:
        translation, json_ok = json.loads(raw)["label"], True
    except Exception:
        translation, json_ok = raw, False
    de_ph = [p.lower() for p in PLACEHOLDER.findall(text)]               # [Andere Gruppe] = [andere Gruppe]
    en_ph = [p.lower() for p in PLACEHOLDER.findall(translation or "")]
    ratio = len(translation or "") / max(len(text), 1)
    return {
        "row_id": row_id,
        "translation": translation,
        "raw_output": raw,
        "json_ok": json_ok,
        "placeholder_ok": sorted(de_ph) == sorted(en_ph),
        "length_ratio": round(ratio, 2),
        "length_ok": 0.75 <= ratio <= 1.25,
        "tokens_prompt": res["tokens_prompt"],
        "tokens_response": res["tokens_response"],
        "seconds": round(time.time() - start, 1),
        "error": res["error"],
        "model": MODEL,
        "seed": SEED,
    }


_lock = threading.Lock()
STOP_AFTER_ERRORS = 20   # pause the run if the server fails this many times in a row

def model_of(rec):
    # rows from before the model was recorded were all made with gemma4-26b
    return rec.get("model") or "gemma4-26b"

def done_ids():
    """row_ids already translated successfully WITH THE CURRENT MODEL (rows from other models get redone)."""
    if not os.path.exists(CHECKPOINT_FILE):
        return set()
    ids = set()
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if rec["error"] is None and rec["translation"] and model_of(rec) == MODEL:
                ids.add(rec["row_id"])
    return ids

def run_batch(rows):
    todo = rows[~rows["row_id"].isin(done_ids())]
    print(f"To translate: {len(todo)} (already done: {len(rows) - len(todo)})")
    if todo.empty:
        return True
    t0, n, n_err, in_a_row = time.time(), 0, 0, 0
    pool = ThreadPoolExecutor(max_workers=MAX_WORKERS)
    futures = [pool.submit(translate_row, r.row_id, r.text_block) for r in todo.itertuples()]
    try:
        for fut in as_completed(futures):
            rec = fut.result()
            with _lock:
                with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n += 1
            n_err += rec["error"] is not None
            in_a_row = in_a_row + 1 if rec["error"] else 0
            if in_a_row >= STOP_AFTER_ERRORS:
                print(f"\n⚠️ {in_a_row} errors in a row: the server seems to be down. Stopping for now.")
                print("Everything finished so far is saved. Run this cell again later to continue.")
                return False
            if n % 50 == 0 or n == len(todo):
                el = time.time() - t0
                eta = el / n * (len(todo) - n)
                print(f"{n}/{len(todo)} done | errors: {n_err} | {el/60:.1f} min so far | ~{eta/60:.0f} min left")
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
    return True

def load_results():
    """Latest successful result per row_id, ONLY from the current model (so the final file uses one model)."""
    recs = [json.loads(l) for l in open(CHECKPOINT_FILE, encoding="utf-8")]
    recs = [r for r in recs if model_of(r) == MODEL and r["error"] is None and r["translation"]]
    for r in recs:
        r["model"] = model_of(r)
    return pd.DataFrame(recs).drop_duplicates("row_id", keep="last")

---
## STEP 4: Test batch (20 paragraphs) ⭐ do this first

**✅ Check before the full run:**
- `errors: 0`
- In the table: `json_ok`, `placeholder_ok` and `length_ok` are mostly **True**
- Read 2–3 translations: do they make sense? Is `[Gruppe 1]` kept?
- Look at the **estimate for 5,000** at the end.

In [ ]:
TEST_N = 20
run_batch(sample.head(TEST_N))

res = load_results()
test = sample.head(TEST_N).merge(res, on="row_id", how="left")
print(f"\nJSON ok: {test.json_ok.mean():.0%} | placeholder ok: {test.placeholder_ok.mean():.0%} | length ok: {test.length_ok.mean():.0%}")
print(f"Average {test.seconds.mean():.1f} s per paragraph -> estimate for 5,000 with {MAX_WORKERS} parallel calls: "
      f"~{test.seconds.mean() * 5000 / MAX_WORKERS / 3600:.1f} hours")
test[["row_id", "text_block", "translation", "json_ok", "placeholder_ok", "length_ratio", "error"]]

---
## STEP 4b: Speed test, find the best number of parallel calls ⚡

**What is parallelization?** Instead of sending one paragraph, waiting for the answer and then sending the next, we send **several paragraphs at the same time**. The server works on them together, so the whole job finishes much faster. This is the same as `parallel_calls` in the professor's `bacchuss_lyaeus()`.

**How many at once?** It depends on how busy the server is. Too few = slow. Too many = the server queues them or gives errors (504). This cell **tries 2, 4, 8, 12 and 16** parallel calls on small batches and measures **paragraphs per minute**.

**👉 What to do:** run the cell (about 5–10 minutes). At the end it sets `MAX_WORKERS` to the fastest value without errors.

💡 The test paragraphs are real paragraphs that still need translating, and they are saved, so no API calls are wasted.


In [ ]:
still_todo = sample[~sample["row_id"].isin(done_ids())]
print(f"Paragraphs still to translate with {MODEL}: {len(still_todo)}\n")

speed = []
start_at = 0
for workers in [2, 4, 8, 12, 16]:
    batch = still_todo.iloc[start_at : start_at + 3 * workers]   # 3 rounds per setting
    start_at += len(batch)
    if batch.empty:
        break
    MAX_WORKERS = workers
    before = len(done_ids())
    t0 = time.time()
    run_batch(batch)
    minutes = (time.time() - t0) / 60
    ok = len(done_ids()) - before
    speed.append({"parallel_calls": workers, "paragraphs": len(batch), "ok": ok,
                  "errors": len(batch) - ok, "per_minute": round(ok / minutes, 1)})
    print(f"  -> {workers:>2} parallel calls: {ok}/{len(batch)} ok, {ok / minutes:.1f} paragraphs per minute\n")

speed = pd.DataFrame(speed)
good = speed[speed.errors == 0]
best = (good if not good.empty else speed).sort_values("per_minute").iloc[-1]
MAX_WORKERS = int(best.parallel_calls)
left = len(sample) - len(done_ids())
print(f"Best: MAX_WORKERS = {MAX_WORKERS} ({best.per_minute} paragraphs/min)")
print(f"Estimate for the remaining {left} paragraphs: ~{left / best.per_minute / 60:.1f} hours")
speed


---
## STEP 5: Full run (all 5,000)

**👉 Only start this when the test batch looks good.** It uses the `MAX_WORKERS` found in STEP 4b.

**Only one model:** it translates every paragraph that has no result **from the current `MODEL`** yet. So paragraphs translated earlier with `gemma4-12b` are done again with `gemma4-26b`, and the final file (STEP 6) contains **only `gemma4-26b`** translations.

It prints progress every 50 paragraphs. You can **stop it any time** (■ button): finished paragraphs are already saved. Run this cell again to continue.

💡 Keep the laptop plugged in and don't let it sleep (a long run can take a few hours).

In [ ]:
run_batch(sample)

---
## STEP 6: Check and save the results

**✅ You should see:** how many paragraphs were translated, plus the problem counts. It saves:
- `sample_5k_translated.xlsx`: to read in Excel
- `sample_5k_translated.jsonl`: the input for the warmth/cold annotation

If there are errors, run **STEP 5 again**: it retries only the missing ones.

In [ ]:
res = load_results()
final = sample.merge(res, on="row_id", how="left")

ok = final["translation"].notna() & final["error"].isna()
print(f"Translated: {ok.sum()} / {len(final)}")
print(f"Missing / errors (run STEP 5 again): {(~ok).sum()}")
done = final[ok]
print(f"JSON not valid:          {(done.json_ok == False).sum()}")
print(f"[Gruppe 1] changed:      {(done.placeholder_ok == False).sum()}")
print(f"Length suspicious:       {(done.length_ok == False).sum()}")
print(f"Tokens used: prompt {final.tokens_prompt.sum():,.0f} | response {final.tokens_response.sum():,.0f}")

final.to_excel(OUTPUT_XLSX, index=False)
final.to_json(OUTPUT_JSONL, orient="records", lines=True, force_ascii=False)
print(f"\nSaved {OUTPUT_XLSX} and {OUTPUT_JSONL}")

# Paragraphs to check by hand:
done[(done.placeholder_ok == False) | (done.length_ok == False)][["row_id", "text_block", "translation", "length_ratio"]]